In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
RESULTS_DIR_PATH = '../' + os.environ.get('RESULTS_FOLDER')

COMPETITION_RESULTS_CSV_PATH = RESULTS_DIR_PATH + 'competition_full/competition/results.csv'
CHOICE_RESULTS_CSV_PATH = RESULTS_DIR_PATH + 'choice_full/choice/results.csv'
FAMILIARITY_CHOICE_RESULTS_CSV_PATH = RESULTS_DIR_PATH + 'familiarity_choice_full/familiarity_choice/results.csv'

#### Functions

In [33]:
def _round_or_nan(value, digits: int):
    return round(float(value), digits) if pd.notna(value) and np.isfinite(value) else np.nan

def _to_number_array(values: pd.Series) -> np.ndarray:
    x = pd.to_numeric(values, errors='coerce')
    x = x[np.isfinite(x)]
    return x.to_numpy(dtype=float)

def _to_boolean_array(values: pd.Series) -> np.ndarray:
    x = pd.to_numeric(values, errors='coerce')
    x = x[np.isfinite(x)]

    arr = x.to_numpy(dtype=float)

    if len(arr) == 0:
        return arr.astype(int)

    unique_values = set(np.unique(arr))
    
    if not unique_values.issubset({0.0, 1.0}):
        raise RuntimeError(f'Boolean metric must contain only 0/1. Found: {unique_values}')

    return arr.astype(int)

def _summary(values: pd.Series, metric_type: str):
    if metric_type == 'boolean':
        arr = _to_boolean_array(values)
    else:
        arr = _to_number_array(values)

    n = len(arr)
    avg = float(np.mean(arr)) if n > 0 else np.nan
    std = float(np.std(arr, ddof=1)) if n > 1 else np.nan

    return arr, n, avg, std

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [34]:
def _two_prop_z_test(
    test: np.ndarray,
    control: np.ndarray,
    alternative: str = "two-sided",
):
    n_test = len(test)
    n_control = len(control)

    x_test = int(np.sum(test))
    x_control = int(np.sum(control))

    p_test = x_test / n_test
    p_control = x_control / n_control

    p_pool = (x_test + x_control) / (n_test + n_control)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_test + 1 / n_control))

    if se == 0:
        return 0.0, 1.0

    z_stat = (p_test - p_control) / se

    if alternative == "two-sided":
        p_value = 2 * stats.norm.sf(abs(z_stat))
    elif alternative == "greater":
        p_value = stats.norm.sf(z_stat)
    elif alternative == "less":
        p_value = stats.norm.cdf(z_stat)
    else:
        raise RuntimeError("alternative must be 'two-sided', 'greater', or 'less'")

    return float(z_stat), float(p_value)

In [35]:
def _boolean_significance(
    control: np.ndarray,
    test: np.ndarray,
    *,
    alternative: str,
    min_n_for_z: int,
    min_expected_for_z: float,
):
    n_control = len(control)
    n_test = len(test)

    if n_control == 0 or n_test == 0:
        return "not_enough_data", np.nan, np.nan

    x_control = int(np.sum(control))
    x_test = int(np.sum(test))

    # rows: test/control
    # columns: success/failure
    table = np.array([
        [x_test, n_test - x_test],
        [x_control, n_control - x_control],
    ])

    expected = stats.contingency.expected_freq(table)

    use_fisher = (
        min(n_control, n_test) < min_n_for_z
        or np.any(expected < min_expected_for_z)
    )

    if use_fisher:
        res = stats.fisher_exact(table, alternative=alternative)
        return "fisher_exact", float(res.statistic), float(res.pvalue)

    z_stat, p_value = _two_prop_z_test(
        test=test,
        control=control,
        alternative=alternative,
    )

    return "two_proportion_z_test", z_stat, p_value

In [36]:
def _permutation_mean_test(
    control: np.ndarray,
    test: np.ndarray,
    *,
    alternative: str,
    n_resamples: int,
    random_state: int,
):
    def statistic(test_sample, control_sample):
        return np.mean(test_sample) - np.mean(control_sample)

    n_test = len(test)
    n_control = len(control)

    total_combinations = math.comb(n_test + n_control, n_test)
    actual_resamples = np.inf if total_combinations <= n_resamples else n_resamples

    try:
        res = stats.permutation_test(
            data=(test, control),
            statistic=statistic,
            permutation_type="independent",
            alternative=alternative,
            n_resamples=actual_resamples,
            vectorized=False,
            rng=np.random.default_rng(random_state),
        )
    except TypeError:
        # Для старых версий SciPy
        res = stats.permutation_test(
            data=(test, control),
            statistic=statistic,
            permutation_type="independent",
            alternative=alternative,
            n_resamples=actual_resamples,
            vectorized=False,
            random_state=random_state,
        )

    return "permutation_mean_test", float(res.statistic), float(res.pvalue)

In [37]:
def _number_significance(
    control: np.ndarray,
    test: np.ndarray,
    *,
    alternative: str,
    small_n_threshold: int,
    n_resamples: int,
    random_state: int,
):
    if len(control) < 2 or len(test) < 2:
        return "not_enough_data", np.nan, np.nan

    if min(len(control), len(test)) < small_n_threshold:
        return _permutation_mean_test(
            control=control,
            test=test,
            alternative=alternative,
            n_resamples=n_resamples,
            random_state=random_state,
        )

    res = stats.ttest_ind(
        test,
        control,
        equal_var=False,
        alternative=alternative,
        nan_policy="omit",
    )

    return "welch_t_test", float(res.statistic), float(res.pvalue)

In [47]:
def group_function(
    group_df: pd.DataFrame,
    condition_column: str,
    condition_control: str,
    metric_columns_types: dict[str, str],
    small_n_number: int = 20,
    min_n_for_z_boolean: int = 30,
    min_expected_for_z_boolean: float = 5.0,
    n_resamples: int = 20000,
    random_state: int = 42
) -> pd.Series:
    if condition_column not in group_df.columns:
        raise RuntimeError(f'Column {condition_column} is not found in group DataFrame.')

    if group_df[group_df[condition_column] == condition_control].empty:
        raise RuntimeError(f'Condition {condition_control} is not found in {condition_column} column.')

    for metric_column in metric_columns_types.keys():
        if metric_column not in group_df.columns:
            raise RuntimeError(f'Metric column {metric} is not found in group DataFrame.')

    result = {}

    control_df = group_df[group_df[condition_column] == condition_control]
    control_stats = {}

    for metric_column, metric_type in metric_columns_types.items():
        control_arr, control_n, control_avg, control_std = _summary(control_df[metric_column], metric_type)

        control_stats[metric_column] = {'arr': control_arr, 'n': control_n, 'avg': control_avg, 'std': control_std, 'type': metric_type}

        digits = 3 if metric_type == 'boolean' else 1

        result[(metric_column, condition_control, 'N')] = int(control_n)
        result[(metric_column, condition_control, 'Avg')] = _round_or_nan(control_avg, digits)
        result[(metric_column, condition_control, 'Std')] = _round_or_nan(control_std, digits)

    for condition in group_df[condition_column].dropna().unique():
        if condition == condition_control:
            continue

        condition_df = group_df[group_df[condition_column] == condition]

        for metric_column, metric_type in metric_columns_types.items():
            test_arr, test_n, test_avg, test_std = _summary(condition_df[metric_column], metric_type)

            control_arr = control_stats[metric_column]['arr']
            control_avg = control_stats[metric_column]['avg']

            digits = 3 if metric_type == 'boolean' else 1

            result[(metric_column, condition, 'N')] = int(test_n)
            result[(metric_column, condition, 'Avg')] = _round_or_nan(test_avg, digits)
            result[(metric_column, condition, 'Std')] = _round_or_nan(test_std, digits)

            diff = test_avg - control_avg if pd.notna(test_avg) and pd.notna(control_avg) else np.nan
            r_diff = 100 * (test_avg / control_avg - 1) if pd.notna(test_avg) and pd.notna(control_avg) and control_avg != 0 else np.nan

            result[(metric_column, condition, 'Diff')] = _round_or_nan(diff, digits)
            result[(metric_column, condition, 'R. Diff')] = _round_or_nan(r_diff, 1)

            if metric_type == 'boolean':
                test_name, stat_value, p_value = _boolean_significance(
                    control=control_arr,
                    test=test_arr,
                    alternative='two-sided',
                    min_n_for_z=min_n_for_z_boolean,
                    min_expected_for_z=min_expected_for_z_boolean,
                )
            else:
                test_name, stat_value, p_value = _number_significance(
                    control=control_arr,
                    test=test_arr,
                    alternative='two-sided',
                    small_n_threshold=small_n_number,
                    n_resamples=n_resamples,
                    random_state=random_state,
                )

            result[(metric_column, condition, 'P-Value')] = _round_or_nan(p_value, 4)

    return pd.Series(result)

In [58]:
def _sort_metric_condition_stat_columns(
    df: pd.DataFrame,
    condition_control: str,
) -> pd.DataFrame:
    if not isinstance(df.columns, pd.MultiIndex):
        return df

    stats_order_control = ['N', 'Avg', 'Std']
    stats_order_test = ['N', 'Avg', 'Std', 'Diff', 'R. Diff', 'P-Value']

    metrics = list(dict.fromkeys(df.columns.get_level_values(0)))
    conditions = list(dict.fromkeys(df.columns.get_level_values(1)))

    ordered_cols = []

    for metric in metrics:
        if condition_control in conditions:
            for stat in stats_order_control:
                col = (metric, condition_control, stat)
                if col in df.columns:
                    ordered_cols.append(col)

        for condition in conditions:
            if condition == condition_control:
                continue
            for stat in stats_order_test:
                col = (metric, condition, stat)
                if col in df.columns:
                    ordered_cols.append(col)

    remaining = [col for col in df.columns if col not in ordered_cols]
    ordered_cols.extend(remaining)

    return df.loc[:, ordered_cols]

In [65]:
def calc(
    df: pd.DataFrame,
    group_columns: list[str],
    condition_column: str,
    condition_control: str,
    metric_columns_types: dict[str, str],
    small_n_number: int = 20,
    min_n_for_z_boolean: int = 30,
    min_expected_for_z_boolean: float = 5.0,
    n_resamples: int = 20000,
    random_state: int = 42
) -> pd.DataFrame:
    for group_column in group_columns:
        if group_column not in df.columns:
            raise RuntimeError(f'{group_column} is not found in DataFrame.')
    
    result = df.groupby(by=group_columns).apply(
        lambda g_df: group_function(
            group_df=g_df,
            condition_column=condition_column,
            condition_control=condition_control,
            metric_columns_types=metric_columns_types,
            small_n_number=small_n_number,
            min_n_for_z_boolean=min_n_for_z_boolean,
            min_expected_for_z_boolean=min_expected_for_z_boolean,
            n_resamples=n_resamples,
            random_state=random_state),
        include_groups=False
    )
    
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    result = _sort_metric_condition_stat_columns(result, condition_control=condition_control)

    return result

## Experiment 1. Competition

#### Data

In [7]:
df = pd.read_csv(COMPETITION_RESULTS_CSV_PATH)

In [8]:
df.isna().sum()

gender           0
model_id         0
bet_1            0
bet_2            0
bet_3            0
bet_4            0
competence       0
experiment_id    0
variation_id     0
condition_id     0
iteration_id     0
dtype: int64

In [11]:
df_agg = df.groupby(by=['experiment_id', 'variation_id', 'model_id', 'iteration_id', 'condition_id']).count().reset_index()

In [13]:
df_agg[
    (df_agg['gender'] != 18) |
    (df_agg['bet_1'] != 18) |
    (df_agg['bet_2'] != 18) |
    (df_agg['bet_3'] != 18) |
    (df_agg['bet_4'] != 18) |
    (df_agg['competence'] != 18)
]

,experiment_id,variation_id,model_id,iteration_id,condition_id,gender,bet_1,bet_2,bet_3,bet_4,competence


#### Analytics

## Experiment 2. Choice

In [14]:
df = pd.read_csv(CHOICE_RESULTS_CSV_PATH)

In [15]:
df.isna().sum()

gender                               0
model_id                             0
would_buy_lottery_ticket             0
chosen_ticket                     4320
requested_lottery_ticket_price       0
experiment_id                        0
variation_id                         0
condition_id                         0
iteration_id                         0
dtype: int64

In [16]:
df_agg = df.groupby(by=['experiment_id', 'variation_id', 'model_id', 'iteration_id', 'condition_id']).count().reset_index()

In [18]:
df_agg[
    (df_agg['condition_id'] == 'choice') &
    (
        (df_agg['gender'] != 36) |
        (df_agg['would_buy_lottery_ticket'] != 36) |
        (df_agg['chosen_ticket'] != 36) |
        (df_agg['requested_lottery_ticket_price'] != 36)
    )
]

,experiment_id,variation_id,model_id,iteration_id,condition_id,gender,would_buy_lottery_ticket,chosen_ticket,requested_lottery_ticket_price


In [19]:
df_agg[
    (df_agg['condition_id'] == 'no_choice') &
    (
        (df_agg['gender'] != 36) |
        (df_agg['would_buy_lottery_ticket'] != 36) |
        (df_agg['chosen_ticket'] != 0) |
        (df_agg['requested_lottery_ticket_price'] != 36)
    )
]

,experiment_id,variation_id,model_id,iteration_id,condition_id,gender,would_buy_lottery_ticket,chosen_ticket,requested_lottery_ticket_price


## Experiment 3. Familiarity & Choice

In [20]:
df = pd.read_csv(FAMILIARITY_CHOICE_RESULTS_CSV_PATH)

In [21]:
df.isna().sum()

gender                              0
model_id                            0
would_buy_lottery_ticket            0
chosen_ticket                    1250
would_exchange_lottery_ticket       0
experiment_id                       0
variation_id                        0
condition_id                        0
iteration_id                        0
dtype: int64

In [22]:
df_agg = df.groupby(by=['experiment_id', 'variation_id', 'model_id', 'iteration_id', 'condition_id']).count().reset_index()

In [25]:
df_agg[
    (
        (df_agg['condition_id'] == 'choice_no_familiarity') |
        (df_agg['condition_id'] == 'choice_familiarity')
    ) &
    (
        (df_agg['gender'] != 5) |
        (df_agg['would_buy_lottery_ticket'] != 5) |
        (df_agg['chosen_ticket'] != 5) |
        (df_agg['would_exchange_lottery_ticket'] != 5)
    )
]

,experiment_id,variation_id,model_id,iteration_id,condition_id,gender,would_buy_lottery_ticket,chosen_ticket,would_exchange_lottery_ticket


In [26]:
df_agg[
    (
        (df_agg['condition_id'] == 'no_choice_no_familiarity') |
        (df_agg['condition_id'] == 'no_choice_familiarity')
    ) &
    (
        (df_agg['gender'] != 5) |
        (df_agg['would_buy_lottery_ticket'] != 5) |
        (df_agg['chosen_ticket'] != 0) |
        (df_agg['would_exchange_lottery_ticket'] != 5)
    )
]

,experiment_id,variation_id,model_id,iteration_id,condition_id,gender,would_buy_lottery_ticket,chosen_ticket,would_exchange_lottery_ticket
